# my-coding-agent — Sample Notebook

Demonstrates the  library: tool registration, single-shot LLM calls, and the full agentic loop.

**Prerequisites:** local OMLX server running at  and the package installed:


## 1. Imports

In [ ]:
import importlib
import my_coding_agent

importlib.reload(my_coding_agent)

from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

## 2. Inspect a tool definition

The  converter turns any typed Python function into an OpenAI-compatible schema.

In [ ]:
import json

# Built-in example tool
schema = tool(ToolsRegistry.get_weather)
print(json.dumps(schema, indent=4))

## 3. Register a custom tool

Define a new function and register it on  at runtime.

In [ ]:
def get_stock_price(ticker: str, currency: str = "USD") -> str:
    """Get the current stock price for a ticker symbol."""
    # stub — replace with a real API call
    prices = {"AAPL": 213.5, "GOOGL": 175.2, "MSFT": 425.0}
    price = prices.get(ticker.upper(), 0.0)
    return f"{ticker.upper()} is trading at {price} {currency}"

# Attach to registry so the LLM can call it
ToolsRegistry.get_stock_price = staticmethod(get_stock_price)

print(json.dumps(tool(get_stock_price), indent=4))

## 4. Single-shot LLM call (no tools)

Use  directly for a plain chat completion.

In [ ]:
llm = LLM()

messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user",   "content": "What is 42 in binary?"}
]

resp = llm.chat_completion(messages)
msg = extract_message(resp)
print(msg["content"])

## 5. Single-shot call with tool use

Pass tools manually and inspect the tool-call resp, then execute it.

In [ ]:
tools = [
    tool(ToolsRegistry.get_weather),
    tool(ToolsRegistry.get_stock_price),
]
print("Tools schemas: ", json.dumps(tools, indent=4))

messages = [
    {"role": "system", "content": "Use the available tools to answer."},
    {"role": "user",   "content": "What is the weather in Tokyo? And what is the price of AAPL stock?"}
]

resp = llm.chat_completion(messages, tools=tools)
msg = extract_message(resp)
print("finish_reason:", extract_finish_reason(resp))
print("tool_calls:", json.dumps(msg.get("tool_calls"), indent=4))

In [ ]:
# Execute the tool calls and get the final answer
messages.append(msg)
tool_results = llm.execute_tool_calls(msg)
messages.extend(tool_results)

final_resp = llm.chat_completion(messages)
print(extract_message(final_resp)["content"])

## 6. Full agentic loop with 

 handles the chat → tool-execute → respond cycle automatically.

In [ ]:
from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Use tools when needed. "
            "Available tools: get_weather(location), get_stock_price(ticker, currency)."
        )
    },
    {
        "role": "user", 
        "content": "What is the weather in London and the price of AAPL stock?"
    }
]

agent = Agent(
    messages=messages,
    tools=tools,
)

final_messages = agent.run(max_steps=5)
print(" Final messages: ", json.dumps(final_messages, indent=4))

# 7. Agentic Bash Execution

In [ ]:
import subprocess
subprocess.run("id", shell=True)      # Linux/macOS
subprocess.run("whoami /priv", shell=True)  # Windows

subprocess.run(
    ["ls", "-la"],   # list, not string
    shell=False
)

In [ ]:
import importlib
import my_coding_agent

importlib.reload(my_coding_agent)

from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason


import subprocess
import os
import json

from my_coding_agent._logging import get_logger
from my_coding_agent.tools import ToolsRegistry, tool


def bash(command: str) -> str:
    """
    Execute a bash command and return its output.
    When working with paths, use absolute paths to avoid issues with the current working directory.
    """
    cwd = os.getcwd()
    command = command.strip()
    result = subprocess.run(
        command,
        cwd=cwd,
        shell=True,
        text=True,
        env=os.environ,
        encoding="utf-8",
        errors="replace",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=30,
    )
    output = {
        "stdout": result.stdout, 
        "returncode": result.returncode, 
        "stderr": result.stderr
    }
    return  json.dumps(output, indent=4)

# Attach to registry so the LLM can call it
ToolsRegistry.bash = staticmethod(bash)

# Execution
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Use tools when needed. Use absolute paths when working with files. You are running in a Macbook Pro."
            "Available tools: bash(command) - executes a bash command and returns its output. "
            f"Current path: {os.getcwd()} "
            f"Current directory contents: {os.listdir(os.getcwd())} "
            f"Current OS: {os.name}, Platform: {os.sys.platform}, User: {os.getlogin()}"
        )
    },
    {
        "role": "user", 
        "content": "Using `git` and `gh` CLI tools, ensure the latest local code changes is committed and pushed to GitHub, with standardized commit messages."
    }
]
tools = [
    tool(ToolsRegistry.bash),
]

print("Initial messages: ", json.dumps(messages, indent=4))
print("Available tools: ", json.dumps(tools, indent=4))
print("")
print("")

agent = Agent(
    messages=messages,
    tools=tools,
)

final_messages = agent.run(max_steps=20)
print("Final messages: ", json.dumps(final_messages, indent=4))

Initial messages:  [
    {
        "role": "system",
        "content": "You are a helpful assistant. Use tools when needed. Use absolute paths when working with files. You are running in a Macbook Pro.Available tools: bash(command) - executes a bash command and returns its output. Current path: /Users/noordeepsingh/Workspace/my-coding-agent Current directory contents: ['.archive', 'uv.lock', 'pyproject.toml', '.claude', 'README.md', 'sample.ipynb', '.gitignore', '.venv', '.python-version', '.git', 'src'] Current OS: posix, Platform: darwin, User: root"
    },
    {
        "role": "user",
        "content": "Using `git` and `gh` CLI tools, ensure the latest local code changes is committed and pushed to GitHub, with standardized commit messages."
    }
]
Available tools:  [
    {
        "type": "function",
        "function": {
            "name": "bash",
            "description": "\n    Execute a bash command and return its output.\n    When working with paths, use absolute paths to